# Imports

In [1]:
import pandas as pd
import numpy as np
from langchain_openai import AzureChatOpenAI
import json
import re
import tqdm
from pathlib import Path
from datetime import datetime as dt

# EDA

In [2]:
base_path = Path().cwd()

In [3]:
# # read data and rename columns
# df = pd.read_csv("data/requirements_full.csv")
# df.columns = ["requirement", "s1", "s2", "s3", "s4", "s5", "s6"]
df = pd.read_excel("data/requirements.xlsx")

In [4]:
# make sure all columns except the first one are binary
for c in df.columns[1:]:
    assert df[c].drop_duplicates().shape[0] == 2

# Drop Sensor

In [5]:
# # drop vihecal speed sensor
# df = df.loc[df["s6"]!=1]
# # df.drop(columns="s6", inplace=True)

# Find Examples

In [95]:
N_EXAMPLES = 3

# collect used examples to exclude from test dataset
indexes_to_drop = []

In [96]:
# collect examples with single fault
examples = {}

for c in df.columns[1:]:
    if df[c].sum() == 0: continue
    examples[c] = []
    i = df.loc[ (df[c] == 1) & (df[df.columns[1:][df.columns[1:]!=c]].sum(axis=1) == 0) ].sample(N_EXAMPLES)
    indexes_to_drop.append(i.index)
    for e in i.values:
        examples[c].append([e[0], "[" + ",".join(map(str, e[1:])) + "]"])

In [97]:
# collect examples with multiple faults
examples_multiple = {}

t = df.loc[df[df.columns[1:]].sum(axis=1) == 2]
idx = t.sample(N_EXAMPLES).index
indexes_to_drop.append(idx)
t = df.iloc[idx, :].values

for r in t:
    c = "&".join(df.columns[1:][r[1:]==1])
    if examples_multiple.get(c) is None:
        examples_multiple[c] = []
    examples_multiple[c].append([r[0], "[" + ",".join(map(str, r[1:])) + "]"])

In [98]:
# add both single and multiple into one place
examples.update(examples_multiple)

In [99]:
# join all examples in a text format to add to prompt
examples_txt = ""

for e1 in examples.values():
    for e2 in e1:
        examples_txt += f"Requirement: {e2[0]}\n"
        examples_txt += f"Vector: {e2[1]}\n"
        examples_txt += "\n"

In [93]:
print("\n".join(examples_txt.split("\n")[:8]))

Requirement: The vehicle's control system must monitor the pedal position in real-time to ensure it matches the expected engine and vehicle speed
Vector: [1,0,0,0,0]

Requirement: The control system connected to the accelerator pedal must be capable of tolerating faults without leading to sudden or unexpected throttle responses
Vector: [1,0,0,0,0]

Requirement: The acceleration control system must use predictive algorithms to smooth out acceleration and deceleration in heavy traffic, reducing the risk of rear-end collisions
Vector: [1,0,0,0,0]


In [101]:
# drop indexes used in examples
indexes_to_drop = np.concatenate(indexes_to_drop)
df.drop(index=indexes_to_drop, inplace=True)

# LLM

In [102]:
from prompts.SystemPrompts import SystemPrompt
from prompts.Sensors import Sensors
from prompts.UserPrompt import UserPrompt

In [103]:
# print(SystemPrompt.format(sensors=Sensors,examples=examples_txt))

In [104]:
# llm
llm = AzureChatOpenAI(
    deployment_name="gpt-35",
    temperature=0.0
)

# Run for all Requirements

In [105]:
def parse_result(res):
    '''Parse the LLM result'''
    pattern = r"\[(.*?)\]"
    vec = re.findall(pattern, res)[0]
    return f"[{vec}]".replace(" ", "")


def run_all_reqs(instance):
    # system prompt
    messages = [
        {'role': 'system',
        'content': SystemPrompt.format(sensors=Sensors,examples=examples_txt)}
    ]

    result = {}

    result["idx"] = instance[0]
    result["requirement"] = instance[1].iloc[0]
    result["true_vector"] = "[" + ",".join(map(str, instance[1].iloc[1:])) + "]"

    # add user prompt
    messages.append({"role":"user", "content":UserPrompt.format(req=result["requirement"])})

    # run LLM
    response = llm.invoke(messages)
    result["ai_response"] = response.content
    result["pred_vector"] = parse_result(result["ai_response"])

    result["accuracy"] = result["pred_vector"] == result["true_vector"]

    result["ai_token_usage"] = response.response_metadata["token_usage"]

    return result

In [106]:
results = [run_all_reqs(i) for i in tqdm.tqdm(df.iterrows())]

209it [03:10,  1.10it/s]


In [107]:
accuracy = 0
total_tokens = 0
total_completion_tokens = 0

for r in results:
    accuracy += r["accuracy"]
    total_tokens += r["ai_token_usage"]["total_tokens"]
    total_completion_tokens += r["ai_token_usage"]["completion_tokens"]

number_of_reqs = len(results)
accuracy /= len(results)
avg_token_per_req = total_tokens / len(results)
avg_completion_token_per_req = total_completion_tokens / len(results)

In [108]:
accuracy

0.8660287081339713

In [109]:
# save results
time = dt.now()

results_path = "results/conv_{model}_n-{examples}_acc-{accuracy}_{time}.json"
results_path = results_path.format(
    model=llm.deployment_name,
    examples=N_EXAMPLES,
    time=time.strftime('%m.%d.%Y-%H:%M:%S'),
    accuracy=round(accuracy, 3)
)

results_file = base_path / results_path
results_file.parent.mkdir(exist_ok=True)
results_file.touch()

with results_file.open("w") as f:
    json.dump({"accuracy": accuracy,
        "number_of_reqs": number_of_reqs,
        "total_tokens": total_tokens,
        "total_completion_tokens": total_completion_tokens,
        "avg_token_per_req": avg_token_per_req,
        "avg_completion_token_per_req": avg_completion_token_per_req,
        "responses": results}, f, indent=4)